# torchrolling on a GPU

Checks that torchrolling is correct on a real GPU, tunes its kernels and benchmarks it
against pandas, polars, cuDF and a plain `torch.unfold`.

**Before running:** Settings → Accelerator → **GPU T4 x2**, and Settings → **Internet on**.
Then **Run All in the editor** (not Save Version), so every cell's output shows up as it
runs. The whole notebook takes about an hour: tests ~15 min, tuning ~15 min, benchmark
~30 min.

In [ ]:
!nvidia-smi

## Install

Clones the repository and installs it on top of Kaggle's own torch (which already has CUDA
and Triton). The commit it prints is the one being tested and measured.

In [ ]:
REPO = "https://github.com/LenaBarretta/torchrolling"
SRC = "/kaggle/working/torchrolling"

# Always a fresh clone and a fresh install, so a rerun never measures older code.
!rm -rf {SRC} && git clone -q {REPO} {SRC} && git -C {SRC} log -1 --format='%h %s'
!pip install -q pytest pytest-cov hypothesis mpmath pandas polars
!pip install -q --no-deps --force-reinstall {SRC}

In [ ]:
import torch, triton, torchrolling
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| triton", triton.__version__)
print("torchrolling", torchrolling.__version__, "|", torch.cuda.get_device_name(0))

Optional: cuDF (RAPIDS), the closest GPU competitor. The install is large and can take a few
minutes; skip this cell to benchmark without it.

In [ ]:
!pip install -q --extra-index-url=https://pypi.nvidia.com cudf-cu12

## Correctness on the GPU

The full test suite. With CUDA available, the `test_cuda_*` tests also run: they compare the
GPU results (including the Triton quantile kernel) with the CPU code path, which itself is
checked against pandas.

In [ ]:
!cd {SRC} && python -m pytest -q -p no:cacheprovider --no-cov tests

## Tune the kernels

Tries tile sizes and warp counts on this GPU and keeps the fastest (about 15 minutes): for the
small-window kernel, and where it stops beating the van Herk kernel; for quantiles with the
largest windows; and for EWM. The van Herk tables in the source were measured on a T4 already.
The benchmark below uses the result.

In [ ]:
!cd {SRC} && python bench/tune.py --parts direct,quantile,ewm

## Benchmark

Methods: rolling mean, std, max, median, corr, and an EWM mean, on float32 data. On CUDA
torchrolling runs twice: with its default float32 accumulator and with
`acc_dtype=torch.float64` (the `fp64 acc` column), which shows what the extra precision
costs on this GPU. The largest case is 10,000 series × 10,000 points; CPU libraries are
skipped above 10 million values (`--max-cpu-elements`).

In [ ]:
!cd {SRC} && python bench/bench.py --tuning bench/results/tuning.json

In [ ]:
from pathlib import Path
from IPython.display import Markdown

results = sorted(Path(SRC, "bench", "results").glob("*.md"))
Markdown("\n\n".join(p.read_text() for p in results))

## Keep the results

Download `bench-results.zip` (Output panel on the right, after the cell below), unpack it
into the repository so that `bench/results/` holds the `.json` and `.md` files, and send
them over: the README table is generated from them.

In [ ]:
!cd {SRC} && zip -qr /kaggle/working/bench-results.zip bench/results && ls -l /kaggle/working/bench-results.zip